# Question 4 — DAO revenue, equilibrium utilization, and curve reconfiguration

**Question:** Quantify annualized DAO revenue on $1B of USDe deposits at the status quo, showing your derivation of equilibrium utilization. Propose at least three alternative curve configurations (slope1, Uopt) that compress the spread between the USDe borrow rate and the sUSDe yield. For each: equilibrium utilization, borrow rate, DAO revenue, supply APY, pros and cons. Two constraints: are compressing the spread and maximizing revenue conflicting objectives (argue with numbers)? Is sUSDe yield independent of Aave's utilization (justify the assumption)?

**Approach:** derive equilibrium utilization from looper behavior — loopers add leverage until ROE falls to a hurdle rate, which pins the borrow rate they'll pay, which the IR curve maps to utilization. Utilization is a derived quantity, not an assumption.

## 4.1 — Parameters, with sources

Contract-sourced figures reuse the pinned block (25,682,519) used throughout — its state is immutable, so re-querying would be redundant.

In [7]:
# --- On-chain, USDe reserve, block 25,682,519 (verified via getReserveConfigurationData / IR strategy contract) ---
RF     = 0.25   # reserve factor
r0     = 0.00   # IR curve base rate
Uopt   = 0.90   # IR curve optimal utilization (kink)
slope1 = 0.04   # IR curve slope1 -- confirmed on-chain value, NOT the 3.5% placeholder used in early drafts
slope2 = 0.12   # IR curve slope2
observed_U = 0.7420  # actual on-chain utilization at the same block, for the sanity check in 4.3

# --- From the loop analysis, on-chain, block 25,682,519 ---
total_usde_debt = 582_621_785   # total USDe variable debt on the reserve
loop_debt       = 295_154_950   # debt held by addresses with both sUSDe collateral and USDe debt (both-legs-nonzero definition)
loop_share      = loop_debt / total_usde_debt
L_debt_weighted = 10.99          # population-wide debt-weighted leverage, all 51 non-dust identified loopers (Section 2.3 of the loop analysis)
L_median        = 11.44          # median leverage across that same broader population

print(f"Loop share of total USDe debt: {loop_share:.1%}")
print(f"Population-wide debt-weighted leverage (51 loopers, Section 2.3): {L_debt_weighted:.2f}x  (median: {L_median:.2f}x)")
print("Shown for context only. The model below instead uses the top-10-by-debt cluster's own debt-weighted")
print("leverage (computed in Section 4.2, alongside the hurdle) rather than this broader population figure --")
print("see Section 4.2 for why, and Section 4.3 for the tradeoff that choice implies.")

Loop share of total USDe debt: 50.7%
Population-wide debt-weighted leverage (51 loopers, Section 2.3): 10.99x  (median: 11.44x)
Shown for context only. The model below instead uses the top-10-by-debt cluster's own debt-weighted
leverage (computed in Section 4.2, alongside the hurdle) rather than this broader population figure --
see Section 4.2 for why, and Section 4.3 for the tradeoff that choice implies.


## 4.2 — Hurdle rate: an explicit, flagged assumption, and why leverage is sourced from the same cluster

The hurdle rate — the minimum ROE a looper requires to keep the position open — isn't observable on-chain and has to be assumed. Rather than guess a buffer, it's grounded in data: the top 10 loopers by debt (Section 2.4 of the loop analysis) have leverage and HF from `getUserAccountData`. Excluding one outlier running a materially different, lower-leverage strategy (2.60x, HF 1.528), the remaining 9 run 8.31x–12.74x at HF 1.014–1.046. Plugging each address's own (leverage, HF) into the loop's ROE identity at today's case rates gives its realized ROE — the hurdle below is the debt-weighted result, used unrounded.

In [8]:
LT = 0.94  # e-mode category 32 liquidation threshold
y_s_case, r_b_case = 0.04, 0.033  # sUSDe yield, actual on-chain borrow rate at observed utilization (Section 4.1)

# Top 10 loopers by debt at block 25,682,519: (address, debt $, leverage, HF) -- leverage and HF both
# read directly from Aave's own getUserAccountData (Section 2.3/2.4 of the loop analysis), not re-derived.
# 0x17b6AA44f45a732487F95b524f15053e8149376F excluded: 2.60x leverage / HF=1.528 is a materially
# different, much lower-leverage strategy -- not a tight loop.
top10_cluster = [
    ("0xCf0a12CBd8088fc5f84ad431E71787157041cD69", 74_068_359,  9.95, 1.023),
    ("0x6142EB927529974c5cDEd66dafc57cB5AaaF73Ab", 50_084_699, 11.73, 1.028),
    ("0x42715bA91deDa3c692B9F540CEe2FBb4daE78bBB", 45_096_733, 12.49, 1.022),
    ("0x086eb9C2B14dD657ae77D07dF9115A2F946CE327", 34_322_313, 12.42, 1.022),
    ("0xd7583E3CF08bbcaB66F1242195227bBf9F865Fda", 14_200_347, 12.74, 1.020),
    ("0x6a73204dB71F8e054bf9A0680b02Ae96f700b595", 10_727_582,  8.31, 1.046),
    ("0x8933850c117bAf7B6423fBB996b4EfA23B67f64a",  7_959_707, 12.19, 1.024),
    ("0xd6c757043e7d59088969B188923C62fa960aFE9B",  4_931_008, 12.72, 1.020),
    ("0x256C75846B4B605aCaF5Cf8b05AE8d239eB298CD",  4_081_523, 12.28, 1.014),
]

total_debt = sum(d[1] for d in top10_cluster)
w_L  = sum(d[1] * d[2] for d in top10_cluster) / total_debt
w_HF = sum(d[1] * d[3] for d in top10_cluster) / total_debt

roes = [(L * y_s_case - (L - 1) * r_b_case) for _, _, L, _ in top10_cluster]
w_roe = sum(debt * (L * y_s_case - (L - 1) * r_b_case) for _, debt, L, _ in top10_cluster) / total_debt

print(f"Debt-weighted leverage across cluster: {w_L:.4f}x")
print(f"Debt-weighted HF across cluster:       {w_HF:.4f}")
print(f"Realized ROE range across cluster:     {min(roes):.2%} - {max(roes):.2%}")
print(f"Debt-weighted realized ROE (hurdle):   {w_roe:.4%}")

print(f"\nCluster = ${total_debt:,.0f} debt, {total_debt/loop_debt:.1%} of all loop-driven USDe debt (9 addresses).")
print("Hurdle and leverage below both come from this same cluster -- consistent, not mixed with the broader population.")

Debt-weighted leverage across cluster: 11.3819x
Debt-weighted HF across cluster:       1.0244
Realized ROE range across cluster:     9.12% - 12.22%
Debt-weighted realized ROE (hurdle):   11.2673%

Cluster = $245,472,271 debt, 83.2% of all loop-driven USDe debt (9 addresses).
Hurdle and leverage below both come from this same cluster -- consistent, not mixed with the broader population.


## 4.3 — The model

Loopers keep adding leverage until their ROE falls to the hurdle rate. Solving that condition for the borrow rate they're indifferent at:

$$\text{ROE} = y_s + (L-1)(y_s - r_b) = \text{hurdle} \implies r_b^* = y_s - \frac{\text{hurdle} - y_s}{L - 1}$$

$r_b^*$ depends only on the loop's own economics (yield, hurdle, leverage) — **not on the IR curve**. The curve's only job is to say *what utilization delivers that rate*:

$$U^* = \begin{cases} U_{opt} \cdot \dfrac{r_b^* - r_0}{\text{slope1}} & r_b^* \le r_0 + \text{slope1} \text{ (below kink)} \\[6pt] U_{opt} + (1-U_{opt}) \cdot \dfrac{r_b^* - r_0 - \text{slope1}}{\text{slope2}} & \text{otherwise (above kink)} \end{cases}$$

Then: $\text{Revenue} = \$1B \cdot U^* \cdot r_b^* \cdot RF$, and $\text{Supply APY} = r_b^* \cdot U^* \cdot (1-RF)$.

In [9]:
def solve_equilibrium(y_s, hurdle, L, r0, Uopt, slope1, slope2, RF, deposits=1e9, verbose=True):
    r_b_star = y_s - (hurdle - y_s) / (L - 1)
    kink_rate = r0 + slope1
    if r_b_star <= kink_rate:
        U_star = Uopt * (r_b_star - r0) / slope1
        branch = "below kink"
    else:
        U_star = Uopt + (1 - Uopt) * (r_b_star - r0 - slope1) / slope2
        branch = "above kink"
    revenue = deposits * U_star * r_b_star * RF
    supply_apy = r_b_star * U_star * (1 - RF)
    if verbose:
        print(f"  r_b* = {y_s:.2%} - ({hurdle:.4%}-{y_s:.2%})/({L:.2f}-1) = {r_b_star:.4%}")
        print(f"  branch: {branch} (kink rate = {kink_rate:.2%})")
        print(f"  U*   = {U_star:.4%}")
        print(f"  Revenue on $1B  = ${revenue:,.0f}")
        print(f"  Supply APY      = {supply_apy:.4%}")
    return dict(r_b_star=r_b_star, U_star=U_star, branch=branch, revenue=revenue, supply_apy=supply_apy)

y_s, hurdle, L = 0.04, w_roe, w_L  # both from Section 4.2's top-10 cluster, unrounded, used consistently
print("=== STATUS QUO ===")
status_quo = solve_equilibrium(y_s, hurdle, L, r0, Uopt, slope1, slope2, RF)

print(f"\nSanity check against observed on-chain utilization ({observed_U:.2%}):")
print(f"  Model says U* = {status_quo['U_star']:.2%}, gap = {status_quo['U_star']-observed_U:+.2%}")
print(f"  For comparison, the same model with the 3.5% placeholder (not the real 4.0% slope1) gives:")
placeholder = solve_equilibrium(y_s, hurdle, L, r0, Uopt, 0.035, slope2, RF, verbose=False)
print(f"    U* = {placeholder['U_star']:.2%}, gap = {placeholder['U_star']-observed_U:+.2%} -- a much worse fit.")
print(f"  Using the real on-chain slope1 doesn't just correct a number, it materially improves model fit.")

=== STATUS QUO ===
  r_b* = 4.00% - (11.2673%-4.00%)/(11.38-1) = 3.3000%
  branch: below kink (kink rate = 4.00%)
  U*   = 74.2500%
  Revenue on $1B  = $6,125,625
  Supply APY      = 1.8377%

Sanity check against observed on-chain utilization (74.20%):
  Model says U* = 74.25%, gap = +0.05%
  For comparison, the same model with the 3.5% placeholder (not the real 4.0% slope1) gives:
    U* = 84.86%, gap = +10.66% -- a much worse fit.
  Using the real on-chain slope1 doesn't just correct a number, it materially improves model fit.


**Caveat:** the 74.25% vs. 74.20% closeness is expected by construction, not proof the model is right — `hurdle` is this cluster's own debt-weighted ROE at leverage 11.38x and the case rate, so feeding it back through `r_b*` algebraically returns that same rate regardless of whether loopers actually behave this way.

What *isn't* circular: the IR curve mapping (r_b* → U*) was never part of building hurdle or leverage, so landing close to observed utilization is a real, checkable statement about the curve — not the looper-behavior theory. Hurdle and leverage are both sourced from the same top-10 cluster (83% of loop-driven debt, Section 4.2) rather than mixed with the broader 51-address population, for internal consistency.

## 4.4 — Sensitivity: bounding DAO revenue under HF and leverage uncertainty

Hurdle (11.27%) and leverage (11.38x) are point estimates from a thin sample (9 addresses). Bound the status quo with one table, hurdle held fixed:

- Three rows target **HF** (1.025 / 1.05 / 1.10, bracketing the cluster's observed 1.014–1.046) and derive leverage via $L = 1/(1-LT/HF)$.
- Three rows target **leverage** directly (10x / 11.38x / 12x) and derive HF in reverse — except the 11.38x row, which shows the cluster's actual measured HF (1.0244), not the formula's implied value (1.0305); real positions drift from the 92% origination cap toward the 94% LT as interest accrues, so they don't sit exactly on the idealized curve.

HF and leverage are the same lever, not independent — shown as one table sorted by revenue, not a 3×3 grid.

In [10]:
LT = 0.94
HURDLE = hurdle  # pulled from Section 4.3 -- the exact debt-weighted realized ROE, held fixed throughout

def L_from_HF(hf, lt=LT):
    return 1 / (1 - lt / hf)

def HF_from_L(L, lt=LT):
    return lt / (1 - 1 / L)

scenarios = []
for hf in [1.025, 1.05, 1.10]:
    L_implied = L_from_HF(hf)
    r = solve_equilibrium(y_s, HURDLE, L_implied, r0, Uopt, slope1, slope2, RF, verbose=False)
    scenarios.append({"Scenario": f"HF = {hf}", "HF": hf, "L": L_implied, **r})

for L_assumed in [10, w_L, 12]:  # w_L = 11.38x, the cluster's exact debt-weighted leverage from Section 4.2
    is_base = abs(L_assumed - w_L) < 1e-9
    hf_display = w_HF if is_base else HF_from_L(L_assumed)  # base row: actual measured HF, not formula-implied
    label = f"L = {L_assumed:.2f}x (base case)" if is_base else f"L = {L_assumed:.0f}x"
    r = solve_equilibrium(y_s, HURDLE, L_assumed, r0, Uopt, slope1, slope2, RF, verbose=False)
    scenarios.append({"Scenario": label, "HF": hf_display, "L": L_assumed, **r})

import pandas as pd

sens_df = pd.DataFrame(scenarios).sort_values("revenue").reset_index(drop=True)
sens_disp = sens_df.copy()
sens_disp["HF"] = sens_disp["HF"].map(lambda x: f"{x:.4f}")
sens_disp["L"] = sens_disp["L"].map(lambda x: f"{x:.2f}x")
sens_disp["r_b_star"] = sens_disp["r_b_star"].map(lambda x: f"{x:.2%}")
sens_disp["U_star"] = sens_disp["U_star"].map(lambda x: f"{x:.2%}")
sens_disp["revenue"] = sens_disp["revenue"].map(lambda x: f"${x/1e6:.2f}M")
sens_disp["supply_apy"] = sens_disp["supply_apy"].map(lambda x: f"{x:.2%}")
sens_disp = sens_disp[["Scenario", "HF", "L", "U_star", "r_b_star", "revenue", "supply_apy"]]
sens_disp.columns = ["Scenario", "HF (measured for base row, else formula-implied)", "Assumed L", "Equilibrium U*", "Borrow rate r_b*", "DAO Revenue", "Supply APY"]
print(f"Combined sensitivity -- hurdle fixed at {HURDLE:.4%}, sorted by revenue")
display(sens_disp)

print(f"\nRange of DAO revenue across all 6 scenarios: ${sens_df['revenue'].min()/1e6:.2f}M - ${sens_df['revenue'].max()/1e6:.2f}M per $1B in loop-driven deposits")
print(f"  ({sens_df['revenue'].min()/1e9*10000:.0f}bps - {sens_df['revenue'].max()/1e9*10000:.0f}bps)")

Combined sensitivity -- hurdle fixed at 11.2673%, sorted by revenue


,Scenario,"HF (measured for base row, else formula-implied)",Assumed L,Equilibrium U*,Borrow rate r_b*,DAO Revenue,Supply APY
0,HF = 1.1,1.1000,6.87x,62.17%,2.76%,$4.29M,1.29%
1,HF = 1.05,1.0500,9.55x,70.87%,3.15%,$5.58M,1.67%
2,L = 10x,1.0444,10.00x,71.83%,3.19%,$5.73M,1.72%
3,L = 11.38x (base case),1.0244,11.38x,74.25%,3.30%,$6.13M,1.84%
4,L = 12x,1.0255,12.00x,75.13%,3.34%,$6.27M,1.88%
5,HF = 1.025,1.0250,12.06x,75.21%,3.34%,$6.29M,1.89%



Range of DAO revenue across all 6 scenarios: $4.29M - $6.29M per $1B in loop-driven deposits
  (43bps - 63bps)


The base case sits mid-range among the six scenarios. Annual DAO revenue on \$1B spans **\$4.29M–\$6.29M** (43–63bps) — this range, not the single point estimate, should carry into a revenue projection given the thin sample (9 addresses, 83% of loop volume).

## 4.5 — Three alternative curve configurations

Each config compresses the spread by reaching the same rate at lower utilization (steeper slope1, lower Uopt, or both); loop demand reacts to the new curve, so each is re-solved as a fixed point, not held constant.

In [11]:
configs = {
    "Status quo":       dict(Uopt=0.90, slope1=0.04),
    "A: lower Uopt":    dict(Uopt=0.80, slope1=0.04),
    "B: raise slope1":  dict(Uopt=0.90, slope1=0.07),
    "C: moderate both": dict(Uopt=0.85, slope1=0.055),
}

rows = []
for name, cfg in configs.items():
    r = solve_equilibrium(y_s, hurdle, L, r0, cfg["Uopt"], cfg["slope1"], slope2, RF, verbose=False)
    rows.append({"config": name, "Uopt": cfg["Uopt"], "slope1": cfg["slope1"], **r})

import pandas as pd
df = pd.DataFrame(rows)
disp = df.copy()
disp["Uopt"] = disp["Uopt"].map(lambda x: f"{x:.0%}")
disp["slope1"] = disp["slope1"].map(lambda x: f"{x:.1%}")
disp["r_b_star"] = disp["r_b_star"].map(lambda x: f"{x:.2%}")
disp["U_star"] = disp["U_star"].map(lambda x: f"{x:.2%}")
disp["revenue"] = disp["revenue"].map(lambda x: f"${x/1e6:.2f}M")
disp["supply_apy"] = disp["supply_apy"].map(lambda x: f"{x:.2%}")
disp = disp[["config","Uopt","slope1","U_star","r_b_star","revenue","supply_apy","branch"]]
disp.columns = ["Config","Uopt","slope1","Equilibrium U*","Borrow rate r_b*","DAO Revenue","Supply APY","Curve branch"]
display(disp)

,Config,Uopt,slope1,Equilibrium U*,Borrow rate r_b*,DAO Revenue,Supply APY,Curve branch
0,Status quo,90%,4.0%,74.25%,3.30%,$6.13M,1.84%,below kink
1,A: lower Uopt,80%,4.0%,66.00%,3.30%,$5.45M,1.63%,below kink
2,B: raise slope1,90%,7.0%,42.43%,3.30%,$3.50M,1.05%,below kink
3,C: moderate both,85%,5.5%,51.00%,3.30%,$4.21M,1.26%,below kink


**Pros and cons:**

- **Status quo:** \$6.13M revenue, 74.25% U, 1.84% supply APY. *Pro:* highest revenue. *Con:* no spread compression.
- **A — lower Uopt to 80%:** \$5.45M (−11.1%), 66.00% U. *Pro:* smallest revenue hit for the compression achieved; one-parameter change. *Con:* also compresses headroom for organic borrowers below the kink.
- **B — raise slope1 to 7.0%:** \$3.50M (−42.9%), 42.43% U. *Pro:* most aggressive compression, nearly halves utilization. *Con:* largest revenue cost; a visible, discontinuous market signal.
- **C — moderate both (Uopt 85%, slope1 5.5%):** \$4.21M (−31.3%), 51.00% U. *Pro:* middle path, less likely to hit either parameter's edge case. *Con:* still a substantial cut, and two parameters to explain/monitor.

## 4.6 — Constraint 1: are compressing the spread and maximizing revenue conflicting objectives?

**Yes, algebraically, not just empirically.** $r_b^*$ is set entirely by the loop's hurdle condition and doesn't depend on the curve — the curve only sets *how much utilization* delivers that rate. Since $\text{Revenue} = RF \cdot U^* \cdot r_b^*$ and $r_b^*$ is fixed, revenue is directly proportional to $U^*$, and any spread-compressing curve change reaches the same rate at lower utilization. A sweep confirms this holds across the whole slope1 range, not just the three sampled configs.

In [12]:
sweep = []
for s1 in [0.02, 0.03, 0.034, 0.04, 0.05, 0.07, 0.10, 0.15, 0.20]:
    r = solve_equilibrium(y_s, hurdle, L, r0, 0.90, s1, slope2, RF, verbose=False)
    sweep.append({"slope1": s1, "U_star": r["U_star"], "revenue_M": r["revenue"]/1e6})

sweep_df = pd.DataFrame(sweep)
sweep_disp = sweep_df.copy()
sweep_disp["slope1"] = sweep_disp["slope1"].map(lambda x: f"{x:.1%}")
sweep_disp["U_star"] = sweep_disp["U_star"].map(lambda x: f"{x:.2%}")
sweep_disp["revenue_M"] = sweep_disp["revenue_M"].map(lambda x: f"${x:.2f}M")
display(sweep_disp)

print("Revenue is monotonically decreasing, not hump-shaped -- holds because slope2 is much steeper than the")
print(f"loop's target rate ({status_quo['r_b_star']:.2%}), so the curve never caps utilization near 100% before reaching it.")

,slope1,U_star,revenue_M
0,2.0%,91.08%,$7.51M
1,3.0%,90.25%,$7.45M
2,3.4%,87.35%,$7.21M
3,4.0%,74.25%,$6.13M
4,5.0%,59.40%,$4.90M
5,7.0%,42.43%,$3.50M
6,10.0%,29.70%,$2.45M
7,15.0%,19.80%,$1.63M
8,20.0%,14.85%,$1.23M


Revenue is monotonically decreasing, not hump-shaped -- holds because slope2 is much steeper than the
loop's target rate (3.30%), so the curve never caps utilization near 100% before reaching it.


## 4.7 — Constraint 2: is sUSDe yield independent of Aave's utilization?

**No.** Treating $y_s$ as a fixed 4.0% is an explicit simplification, not a claim of fact.

**Mechanism:** $y_s = Y_{portfolio}/S_{staked}$. The loop mechanically grows $S_{staked}$ (every pass restakes borrowed USDe), so holding $Y_{portfolio}$ fixed in the short run, more staking dilutes $y_s$ — the more the loop scales, the more it erodes its own yield.

**Evidence:** sUSDe's 7-day APY fell from 9.4% (April 2026) to 7.1% (June 2026) as funding compressed. Ethena's founder has publicly described the inverse mechanism (falling yield → unstaking → $y_s$ recovers) as an intentional design feature ([The Block](https://www.theblock.co/amp/post/278802/ethena-eth-perpetual-futures-open-interest-risk)) — confirming the coupling is real enough to design around.

**Why the model still fixes 4.0%:** the brief's case inputs anchor status quo to 4.0%, and modeling the endogenous feedback would need assumptions about its shape/speed that aren't sourced here. The figures above are a first-order approximation likely to overstate how much the loop can scale before eroding its own economics.

## 4.8 — Summary for the memo

- **Status quo:** equilibrium utilization **74.25%**, borrow rate **3.30%**, DAO revenue on \$1B **\$6.13M/yr**, supply APY **1.84%**. Uses the top-10 cluster's own hurdle (11.27%) and leverage (11.38x) — 0.05 points from observed utilization (74.20%), expected by construction (4.3), not independent proof.
- **Sensitivity bound:** six HF/leverage scenarios put annual revenue on \$1B between **\$4.29M–\$6.29M** (43–63bps) — the range to carry forward, given the thin 9-address sample.
- **Three configs tested** all compress the spread and cost revenue: −11.1%, −42.9%, −31.3%, against utilization drops of 8.25, 31.82, 23.25 points.
- **Spread compression and revenue maximization directly conflict**, algebraically: the loop's target rate doesn't respond to curve shape, so revenue tracks utilization one-for-one, and utilization is exactly what compression reduces.
- **sUSDe yield is not independent of utilization** — the loop dilutes its own yield source as it scales; the fixed-4.0% assumption is a flagged simplification that likely overstates sustainable loop size.